# Project Sanjay MK2 -- Day 4: Thermal YOLO Training

**Goal:** Train a thermal-specific YOLO11s model on aerial IR imagery. This is the second half of the sensor-adaptive perception architecture (RGB model already trained as `police_full_v2.pt`).

**Prerequisite for:** SensorScheduler RL training (needs both RGB and thermal models).

| Item | Detail |
|------|--------|
| Dataset | HIT-UAV (2,866 aerial thermal IR images, 60-130m altitude) |
| Classes trained | person (0), vehicle (2) -- other classes unused but slot-compatible with RGB model |
| Init | yolo11s.pt (COCO-pretrained, NOT police_full_v2 -- RGB->thermal domain shift) |
| Training | 150 epochs, thermal-specific augmentations |
| Output | thermal_police.pt at Drive/SanjayMK2/runs/thermal_police/weights/ |
| Auto-resume | yes (same pattern as Day 3) |

## CELL 1 -- Install + clone repo

In [ ]:
!pip install ultralytics kagglehub -q
!git clone https://github.com/aniket-27507/Sanjay_MK2 /content/Sanjay_MK2
%cd /content/Sanjay_MK2

# Write thermal dataset YAML (same 6-class scheme as RGB for downstream compatibility)
import os
os.makedirs('config/training', exist_ok=True)
with open('config/training/thermal_police.yaml', 'w') as f:
    f.write(
        "path: /content/Sanjay_MK2/data/thermal_training\n"
        "train: images/train\n"
        "val: images/val\n"
        "names:\n"
        "  0: person\n"
        "  1: weapon_person\n"
        "  2: vehicle\n"
        "  3: fire\n"
        "  4: explosive_device\n"
        "  5: crowd\n"
    )
print('Thermal YAML written')

# Verify GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## CELL 2 -- Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Verify Drive structure
drive_path = '/content/drive/MyDrive/SanjayMK2'
os.makedirs(f'{drive_path}/runs', exist_ok=True)
print(f'Drive SanjayMK2 dir: {drive_path}')

## CELL 3 -- Download HIT-UAV + remap to police classes

HIT-UAV classes (source): 0=Person, 1=Car, 2=Bicycle, 3=OtherVehicle, 4=DontCare

Remapped to police scheme:
- 0 (Person) -> 0 (person)
- 1/2/3 (Car/Bicycle/OtherVehicle) -> 2 (vehicle)
- 4 (DontCare) -> dropped

Tested locally: produces 2008 train + 858 val images, 12312 person + 12439 vehicle boxes.

In [ ]:
import kagglehub, shutil
from pathlib import Path

DST = Path('data/thermal_training')

# Skip if already processed
if (DST/'images'/'train').exists() and len(list((DST/'images'/'train').glob('*'))) > 1500:
    n_train = len(list((DST/'images'/'train').glob('*')))
    n_val = len(list((DST/'images'/'val').glob('*')))
    print(f'Thermal training set already exists: {n_train} train + {n_val} val. Skipping.')
else:
    print('Downloading HIT-UAV from Kaggle...')
    src_root = Path(kagglehub.dataset_download('pandrii000/hituav-a-highaltitude-infrared-thermal-dataset'))
    print(f'  Downloaded to: {src_root}')

    # HIT-UAV ships under a 'hit-uav/' subdirectory
    src = src_root / 'hit-uav'
    if not src.exists():
        src = src_root  # fallback if structure changes

    # Remap: HIT-UAV class ID -> police scheme
    REMAP = {0: 0, 1: 2, 2: 2, 3: 2}  # 4 (DontCare) dropped

    counts = {'train': 0, 'val': 0}
    box_counts = {}

    # Merge test into val
    for src_split in ['train', 'val', 'test']:
        dst_split = 'val' if src_split in ('val', 'test') else 'train'
        src_img = src / 'images' / src_split
        src_lbl = src / 'labels' / src_split
        dst_img = DST / 'images' / dst_split
        dst_lbl = DST / 'labels' / dst_split
        dst_img.mkdir(parents=True, exist_ok=True)
        dst_lbl.mkdir(parents=True, exist_ok=True)

        if not src_lbl.exists():
            continue

        for lbl_path in src_lbl.glob('*.txt'):
            # Find matching image
            img_path = None
            for ext in ['.jpg', '.jpeg', '.png', '.bmp']:
                candidate = src_img / (lbl_path.stem + ext)
                if candidate.exists():
                    img_path = candidate
                    break
            if not img_path:
                continue

            # Remap classes, drop DontCare
            lines = []
            for line in lbl_path.read_text().splitlines():
                parts = line.strip().split()
                if len(parts) >= 5:
                    old = int(parts[0])
                    new = REMAP.get(old)
                    if new is not None:
                        parts[0] = str(new)
                        lines.append(' '.join(parts))
                        box_counts[new] = box_counts.get(new, 0) + 1

            if not lines:
                continue

            dst_i = dst_img / f'hituav_{img_path.name}'
            dst_l = dst_lbl / f'hituav_{lbl_path.stem}.txt'
            if not dst_i.exists():
                shutil.copy2(img_path, dst_i)
            if not dst_l.exists():
                dst_l.write_text('\n'.join(lines) + '\n')
            counts[dst_split] += 1

    print(f'\nProcessed:')
    for s, n in counts.items():
        print(f'  {s}: {n} images')
    print(f'Box counts: {box_counts}')
    print(f'  (class 0 = person, class 2 = vehicle)')

## CELL 4 -- Sanity check: dataset structure

In [ ]:
from pathlib import Path
from collections import Counter

DST = Path('data/thermal_training')
for split in ['train', 'val']:
    img_dir = DST / 'images' / split
    lbl_dir = DST / 'labels' / split
    n_img = len(list(img_dir.glob('*'))) if img_dir.exists() else 0
    n_lbl = len(list(lbl_dir.glob('*.txt'))) if lbl_dir.exists() else 0
    c = Counter()
    if lbl_dir.exists():
        for lp in lbl_dir.glob('*.txt'):
            for line in lp.read_text().splitlines():
                parts = line.strip().split()
                if len(parts) >= 5:
                    c[int(parts[0])] += 1
    print(f'{split}: {n_img} images, {n_lbl} labels, classes={dict(sorted(c.items()))}')

print('\nExpected: train ~2008 images, val ~858 images')
print('Expected classes: {0: ~12312, 2: ~12439}')

## CELL 5 -- Train thermal YOLO11s (150 epochs, auto-resumable)

Key differences from RGB training:
- **Init:** `yolo11s.pt` (COCO-pretrained), NOT police_full_v2. RGB->thermal is a hard domain shift; first-layer RGB filters are counterproductive.
- **No HSV augmentation** (`hsv_h=0, hsv_s=0`) -- hue/saturation are meaningless on grayscale IR
- **No vertical flip** (`flipud=0`) -- aerial thermal has gravity; vertical flips hurt
- **More epochs** (150 vs 75) -- thermal needs more training due to lower feature diversity
- **Auto-resume** -- same pattern as Day 3

Training time: ~5-7 hours on T4 (2,866 images, 150 epochs).

In [ ]:
import os
from ultralytics import YOLO

DRIVE_PROJECT = '/content/drive/MyDrive/SanjayMK2/runs'
RUN_NAME = 'thermal_police'
LAST_PT = f'{DRIVE_PROJECT}/{RUN_NAME}/weights/last.pt'

if os.path.exists(LAST_PT):
    print(f'Resuming from: {LAST_PT}')
    print(f'  Checkpoint size: {os.path.getsize(LAST_PT) / 1e6:.1f} MB')
    model = YOLO(LAST_PT)
    results = model.train(resume=True)
else:
    print('Starting fresh thermal training from yolo11s.pt (COCO-pretrained)')
    # Use repo-provided yolo11s.pt; fall back to auto-download
    init_weights = 'yolo11s.pt' if os.path.exists('yolo11s.pt') else 'yolo11s.pt'
    model = YOLO(init_weights)
    results = model.train(
        data='config/training/thermal_police.yaml',
        epochs=150,
        imgsz=640,
        device=0,
        batch=-1,
        patience=30,
        name=RUN_NAME,
        project=DRIVE_PROJECT,
        exist_ok=True,
        # Thermal-specific augmentations
        hsv_h=0.0,      # no hue (grayscale)
        hsv_s=0.0,      # no saturation (grayscale)
        hsv_v=0.3,      # gain variation OK
        mosaic=1.0,
        mixup=0.1,
        fliplr=0.5,     # horizontal flip OK
        flipud=0.0,     # NO vertical flip (aerial has gravity)
        degrees=10.0,
        scale=0.5,
        translate=0.1,
    )

print('\n=== Training complete ===')
print('Best mAP50:', results.results_dict.get('metrics/mAP50(B)'))
print(f'Weights on Drive: {DRIVE_PROJECT}/{RUN_NAME}/weights/')

## CELL 6 -- Download best.pt

After training completes, download `best.pt` and copy locally to:
```
runs/detect/runs/detect/thermal_police/weights/best.pt
```

This becomes the thermal detection backend for the SensorScheduler (Day 5).

In [ ]:
from google.colab import files
import os

best_path = '/content/drive/MyDrive/SanjayMK2/runs/thermal_police/weights/best.pt'
if os.path.exists(best_path):
    print(f'Best weights: {os.path.getsize(best_path) / 1e6:.1f} MB')
    files.download(best_path)
else:
    print(f'WARNING: {best_path} not found')

print()
print('Next steps:')
print('  1. Copy best.pt locally to: runs/detect/runs/detect/thermal_police/weights/best.pt')
print('  2. Day 5: implement src/single_drone/sensor_scheduler.py')

## Day 4 Success Criteria

| Class | Target mAP50 | Notes |
|-------|--------------|-------|
| person | > 0.60 | HIT-UAV has 12K+ person boxes, should train well |
| vehicle | > 0.50 | Merged from Car/Bicycle/OtherVehicle -- mixed quality |
| overall | > 0.55 | 2-class model, should be high |

## What this model is for

- **Primary detection at night** (when RGB is dimmed to 5 FPS)
- **Triggered detection during day** in occlusion scenarios (foliage, tinted vehicles, fire behind walls)
- **Cross-modal verification** when RGB confidence drops

## What this model is NOT

- Not a replacement for police_full_v2 (RGB handles all 6 classes)
- Not trained on weapon_person, fire, explosive_device, crowd (no thermal training data for these)
- Not a full-class detector -- output classes 0 and 2 only have supervision

## Notes on thermal fire

FLAME 2 thermal fire dataset requires Kaggle consent form. Skipped for v1.
Fire detection stays primarily on RGB (police_full_v2 scored 0.762 mAP50 on fire).
Thermal fire can be added later by:
1. Accepting FLAME 2 consent on Kaggle
2. Converting segmentation masks to bounding boxes
3. Adding to training set and retraining